#Import all the requirements

In [122]:
import pandas as pd
import numpy as np
import plotly as pt

#For OS
import os


In [11]:
application_test = pd.read_csv("data/raw/application_test.csv")
application_train = pd.read_csv("data/raw/application_train.csv")
bureau_balance = pd.read_csv("data/raw/bureau_balance.csv")
bureau = pd.read_csv("data/raw/bureau.csv")
credit_crd_balance = pd.read_csv("data/raw/credit_card_balance.csv")
column_description = pd.read_csv("data/raw/HomeCredit_columns_description.csv",encoding = "latin1")
installments = pd.read_csv("data/raw/installments_payments.csv")
Pos_cash_balance = pd.read_csv("data/raw/POS_CASH_balance.csv")
previous_application = pd.read_csv("data/raw/previous_application.csv")
sample_submission = pd.read_csv("data/raw/sample_submission.csv")

In [12]:
application_test.head()

,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100001,Cash loans,F,N,Y,0,135000.0,568800.0,20560.5,450000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,100005,Cash loans,M,N,Y,0,99000.0,222768.0,17370.0,180000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
2,100013,Cash loans,M,Y,Y,0,202500.0,663264.0,69777.0,630000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,4.0
3,100028,Cash loans,F,N,Y,2,315000.0,1575000.0,49018.5,1575000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
4,100038,Cash loans,M,Y,N,1,180000.0,625500.0,32067.0,625500.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
application_test.shape

(48744, 121)

In [14]:
print(application_train.head())
print(application_train.shape)
print(application_train.info)

   SK_ID_CURR  TARGET NAME_CONTRACT_TYPE CODE_GENDER FLAG_OWN_CAR  \
0      100002       1         Cash loans           M            N   
1      100003       0         Cash loans           F            N   
2      100004       0    Revolving loans           M            Y   
3      100006       0         Cash loans           F            N   
4      100007       0         Cash loans           M            N   

  FLAG_OWN_REALTY  CNT_CHILDREN  AMT_INCOME_TOTAL  AMT_CREDIT  AMT_ANNUITY  \
0               Y             0          202500.0    406597.5      24700.5   
1               N             0          270000.0   1293502.5      35698.5   
2               Y             0           67500.0    135000.0       6750.0   
3               Y             0          135000.0    312682.5      29686.5   
4               Y             0          121500.0    513000.0      21865.5   

   ...  FLAG_DOCUMENT_18 FLAG_DOCUMENT_19 FLAG_DOCUMENT_20 FLAG_DOCUMENT_21  \
0  ...                 0             

In [15]:
master = application_train.copy()

In [16]:
master.duplicated().sum()

np.int64(0)

In [17]:
master["SK_ID_CURR"].nunique()

307511

In [18]:
master["SK_ID_CURR"].duplicated().sum()

np.int64(0)

In [19]:
master.shape

(307511, 122)

#File : Bureau_balance

In [20]:
bureau_balance.head()

,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C


In [21]:
bureau_balance.shape

(27299925, 3)

In [22]:
bureau_balance.info()

<class 'pandas.DataFrame'>
RangeIndex: 27299925 entries, 0 to 27299924
Data columns (total 3 columns):
 #   Column          Dtype
---  ------          -----
 0   SK_ID_BUREAU    int64
 1   MONTHS_BALANCE  int64
 2   STATUS          str  
dtypes: int64(2), str(1)
memory usage: 624.8 MB


In [23]:
bureau_balance.shape

(27299925, 3)

In [24]:
bureau_balance.duplicated().sum()

np.int64(0)

In [25]:
#Aggregate due one to many
bureau_balance_agg = (
    bureau_balance
    .groupby("SK_ID_BUREAU")
    .agg(
        bureau_month_count=("MONTHS_BALANCE", "count")
    )
    .reset_index()
)

print(bureau_balance_agg.head(10))
print(bureau_balance_agg.shape)
print(bureau_balance_agg.duplicated().sum())
print(bureau_balance_agg["SK_ID_BUREAU"].duplicated())

   SK_ID_BUREAU  bureau_month_count
0       5001709                  97
1       5001710                  83
2       5001711                   4
3       5001712                  19
4       5001713                  22
5       5001714                  15
6       5001715                  60
7       5001716                  86
8       5001717                  22
9       5001718                  39
(817395, 2)
0
0         False
1         False
2         False
3         False
4         False
          ...  
817390    False
817391    False
817392    False
817393    False
817394    False
Name: SK_ID_BUREAU, Length: 817395, dtype: bool


In [26]:
#Merge with bureau
bureau_with_balance = bureau.merge(
    bureau_balance_agg,
    on="SK_ID_BUREAU",
    how="left"
)

In [27]:
#Aggregate to applicant level
bureau_balance_final = (
    bureau_with_balance
    .groupby("SK_ID_CURR")
    .agg(
        bureau_month_count=("bureau_month_count", "sum")
    )
    .reset_index()
)

In [28]:
#Merge into master
master = master.merge(
    bureau_balance_final,
    on="SK_ID_CURR",
    how="left"
)

In [29]:
print(master["SK_ID_CURR"].duplicated().sum())

0


In [30]:
#File : bureau
print(bureau.head(5))

   SK_ID_CURR  SK_ID_BUREAU CREDIT_ACTIVE CREDIT_CURRENCY  DAYS_CREDIT  \
0      215354       5714462        Closed      currency 1         -497   
1      215354       5714463        Active      currency 1         -208   
2      215354       5714464        Active      currency 1         -203   
3      215354       5714465        Active      currency 1         -203   
4      215354       5714466        Active      currency 1         -629   

   CREDIT_DAY_OVERDUE  DAYS_CREDIT_ENDDATE  DAYS_ENDDATE_FACT  \
0                   0               -153.0             -153.0   
1                   0               1075.0                NaN   
2                   0                528.0                NaN   
3                   0                  NaN                NaN   
4                   0               1197.0                NaN   

   AMT_CREDIT_MAX_OVERDUE  CNT_CREDIT_PROLONG  AMT_CREDIT_SUM  \
0                     NaN                   0         91323.0   
1                     NaN         

In [31]:
print(bureau.shape)


(1716428, 17)


In [32]:
print(bureau.info())


<class 'pandas.DataFrame'>
RangeIndex: 1716428 entries, 0 to 1716427
Data columns (total 17 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_CURR              int64  
 1   SK_ID_BUREAU            int64  
 2   CREDIT_ACTIVE           str    
 3   CREDIT_CURRENCY         str    
 4   DAYS_CREDIT             int64  
 5   CREDIT_DAY_OVERDUE      int64  
 6   DAYS_CREDIT_ENDDATE     float64
 7   DAYS_ENDDATE_FACT       float64
 8   AMT_CREDIT_MAX_OVERDUE  float64
 9   CNT_CREDIT_PROLONG      int64  
 10  AMT_CREDIT_SUM          float64
 11  AMT_CREDIT_SUM_DEBT     float64
 12  AMT_CREDIT_SUM_LIMIT    float64
 13  AMT_CREDIT_SUM_OVERDUE  float64
 14  CREDIT_TYPE             str    
 15  DAYS_CREDIT_UPDATE      int64  
 16  AMT_ANNUITY             float64
dtypes: float64(8), int64(6), str(3)
memory usage: 222.6 MB
None


In [33]:
print(bureau.duplicated().sum())
print(bureau["SK_ID_CURR"].duplicated().sum())

0
1410617


In [34]:
bureau_agg = (
    bureau
    .groupby("SK_ID_CURR")
    .agg(
        bureau_loan_count=("SK_ID_BUREAU", "count"),
        bureau_credit_sum=("AMT_CREDIT_SUM", "sum"),
        bureau_credit_mean=("AMT_CREDIT_SUM", "mean"),
        bureau_debt_sum=("AMT_CREDIT_SUM_DEBT", "sum"),
        bureau_overdue_sum=("AMT_CREDIT_SUM_OVERDUE", "sum")
    )
    .reset_index()
)
print(bureau_agg.head())

   SK_ID_CURR  bureau_loan_count  bureau_credit_sum  bureau_credit_mean  \
0      100001                  7        1453365.000       207623.571429   
1      100002                  8         865055.565       108131.945625   
2      100003                  4        1017400.500       254350.125000   
3      100004                  2         189037.800        94518.900000   
4      100005                  3         657126.000       219042.000000   

   bureau_debt_sum  bureau_overdue_sum  
0         596686.5                 0.0  
1         245781.0                 0.0  
2              0.0                 0.0  
3              0.0                 0.0  
4         568408.5                 0.0  


In [35]:
# Merge bureau
master = master.merge(
    bureau_agg,
    on="SK_ID_CURR",
    how="left"
)

In [36]:
print(master.shape)
print(master["SK_ID_CURR"].duplicated().sum())

(307511, 128)
0


#File 4: credit_crd_balance

In [37]:
credit_crd_balance.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,...,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,...,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,...,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0
3,1389973,337855,-4,236572.110,225000,2250.0,2250.0,0.0,0.0,11795.760,...,233048.970,233048.970,1.0,1,0.0,0.0,10.0,Active,0,0
4,1891521,126868,-1,453919.455,450000,0.0,11547.0,0.0,11547.0,22924.890,...,453919.455,453919.455,0.0,1,0.0,1.0,101.0,Active,0,0


In [38]:
credit_crd_balance.info()

<class 'pandas.DataFrame'>
RangeIndex: 3840312 entries, 0 to 3840311
Data columns (total 23 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   SK_ID_PREV                  int64  
 1   SK_ID_CURR                  int64  
 2   MONTHS_BALANCE              int64  
 3   AMT_BALANCE                 float64
 4   AMT_CREDIT_LIMIT_ACTUAL     int64  
 5   AMT_DRAWINGS_ATM_CURRENT    float64
 6   AMT_DRAWINGS_CURRENT        float64
 7   AMT_DRAWINGS_OTHER_CURRENT  float64
 8   AMT_DRAWINGS_POS_CURRENT    float64
 9   AMT_INST_MIN_REGULARITY     float64
 10  AMT_PAYMENT_CURRENT         float64
 11  AMT_PAYMENT_TOTAL_CURRENT   float64
 12  AMT_RECEIVABLE_PRINCIPAL    float64
 13  AMT_RECIVABLE               float64
 14  AMT_TOTAL_RECEIVABLE        float64
 15  CNT_DRAWINGS_ATM_CURRENT    float64
 16  CNT_DRAWINGS_CURRENT        int64  
 17  CNT_DRAWINGS_OTHER_CURRENT  float64
 18  CNT_DRAWINGS_POS_CURRENT    float64
 19  CNT_INSTALMENT_MATURE_CUM   floa

In [39]:
print(credit_crd_balance.shape)
print(credit_crd_balance["SK_ID_CURR"].nunique())
print(credit_crd_balance["SK_ID_CURR"].duplicated().sum())
print(credit_crd_balance.isnull().sum())

(3840312, 23)
103558
3736754
SK_ID_PREV                         0
SK_ID_CURR                         0
MONTHS_BALANCE                     0
AMT_BALANCE                        0
AMT_CREDIT_LIMIT_ACTUAL            0
AMT_DRAWINGS_ATM_CURRENT      749816
AMT_DRAWINGS_CURRENT               0
AMT_DRAWINGS_OTHER_CURRENT    749816
AMT_DRAWINGS_POS_CURRENT      749816
AMT_INST_MIN_REGULARITY       305236
AMT_PAYMENT_CURRENT           767988
AMT_PAYMENT_TOTAL_CURRENT          0
AMT_RECEIVABLE_PRINCIPAL           0
AMT_RECIVABLE                      0
AMT_TOTAL_RECEIVABLE               0
CNT_DRAWINGS_ATM_CURRENT      749816
CNT_DRAWINGS_CURRENT               0
CNT_DRAWINGS_OTHER_CURRENT    749816
CNT_DRAWINGS_POS_CURRENT      749816
CNT_INSTALMENT_MATURE_CUM     305236
NAME_CONTRACT_STATUS               0
SK_DPD                             0
SK_DPD_DEF                         0
dtype: int64


In [40]:
credit_crd_balance_agg = (
    credit_crd_balance
    .groupby("SK_ID_CURR")
    .agg(
        cc_record_count=("SK_ID_PREV", "count"),
        cc_balance_mean=("AMT_BALANCE", "mean"),
        cc_balance_max=("AMT_BALANCE", "max"),
        cc_limit_mean=("AMT_CREDIT_LIMIT_ACTUAL", "mean"),
        cc_payment_mean=("AMT_PAYMENT_TOTAL_CURRENT", "mean")
    )
    .reset_index()
)

In [41]:
# Cross verification
credit_crd_balance_agg.shape
credit_crd_balance_agg["SK_ID_CURR"].nunique()
credit_crd_balance_agg["SK_ID_CURR"].duplicated().sum()

np.int64(0)

In [42]:
#Credit card balance merge
master = master.merge(
    credit_crd_balance_agg,
    on="SK_ID_CURR",
    how="left"
)

In [43]:
print(master["SK_ID_CURR"].duplicated().sum())

0


File : installments_payments

In [44]:
installments.head()

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585


In [45]:
installments.info()

<class 'pandas.DataFrame'>
RangeIndex: 13605401 entries, 0 to 13605400
Data columns (total 8 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_PREV              int64  
 1   SK_ID_CURR              int64  
 2   NUM_INSTALMENT_VERSION  float64
 3   NUM_INSTALMENT_NUMBER   int64  
 4   DAYS_INSTALMENT         float64
 5   DAYS_ENTRY_PAYMENT      float64
 6   AMT_INSTALMENT          float64
 7   AMT_PAYMENT             float64
dtypes: float64(5), int64(3)
memory usage: 830.4 MB


In [46]:
installments.shape

(13605401, 8)

In [47]:
installments.duplicated().sum()

np.int64(0)

In [48]:
installments["SK_ID_CURR"].duplicated().sum()

np.int64(13265814)

In [49]:
# feature for Payment delay
installments["PAYMENT_DELAY"] = (
    installments["DAYS_ENTRY_PAYMENT"]
    - installments["DAYS_INSTALMENT"]
)

In [50]:
installments_agg = (
    installments
    .groupby("SK_ID_CURR")
    .agg(
        installment_count=("SK_ID_PREV", "count"),
        payment_total=("AMT_PAYMENT", "sum"),
        payment_mean=("AMT_PAYMENT", "mean"),
        payment_max=("AMT_PAYMENT", "max"),
        installment_amount_mean=("AMT_INSTALMENT", "mean"),
        payment_delay_mean=("PAYMENT_DELAY", "mean"),
        payment_delay_max=("PAYMENT_DELAY", "max")
    )
    .reset_index()
)


In [51]:
print(installments_agg.shape)
print(installments_agg["SK_ID_CURR"].nunique())
print(installments_agg["SK_ID_CURR"].duplicated().sum())

(339587, 8)
339587
0


In [52]:
#merge installments
master = master.merge(
    installments_agg,
    on="SK_ID_CURR",
    how="left"
)

In [53]:
print(master["SK_ID_CURR"].duplicated().sum())

0


# Pos_Cash_balance

In [54]:
Pos_cash_balance.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


In [55]:
Pos_cash_balance.shape

(10001358, 8)

In [56]:
Pos_cash_balance.info()

<class 'pandas.DataFrame'>
RangeIndex: 10001358 entries, 0 to 10001357
Data columns (total 8 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   SK_ID_PREV             int64  
 1   SK_ID_CURR             int64  
 2   MONTHS_BALANCE         int64  
 3   CNT_INSTALMENT         float64
 4   CNT_INSTALMENT_FUTURE  float64
 5   NAME_CONTRACT_STATUS   str    
 6   SK_DPD                 int64  
 7   SK_DPD_DEF             int64  
dtypes: float64(2), int64(5), str(1)
memory usage: 610.4 MB


In [57]:
Pos_cash_balance.duplicated().sum()

np.int64(0)

In [58]:
Pos_cash_balance["SK_ID_CURR"].nunique()

337252

In [59]:
Pos_cash_balance.isnull().sum()

SK_ID_PREV                   0
SK_ID_CURR                   0
MONTHS_BALANCE               0
CNT_INSTALMENT           26071
CNT_INSTALMENT_FUTURE    26087
NAME_CONTRACT_STATUS         0
SK_DPD                       0
SK_DPD_DEF                   0
dtype: int64

In [60]:
pos_agg = (
    Pos_cash_balance
    .groupby("SK_ID_CURR")
    .agg(
        pos_record_count=("SK_ID_PREV", "count"),
        pos_months_count=("MONTHS_BALANCE", "count"),
        pos_installment_mean=("CNT_INSTALMENT", "mean"),
        pos_future_installment_mean=("CNT_INSTALMENT_FUTURE", "mean"),
        pos_dpd_mean=("SK_DPD", "mean"),
        pos_dpd_max=("SK_DPD", "max"),
        pos_dpd_def_mean=("SK_DPD_DEF", "mean"),
        pos_dpd_def_max=("SK_DPD_DEF", "max")
    )
    .reset_index()
)

In [61]:
print(pos_agg.shape)
print(pos_agg["SK_ID_CURR"].nunique())
print(pos_agg["SK_ID_CURR"].duplicated().sum())

(337252, 9)
337252
0


In [62]:
#Merge POS file
master = master.merge(
    pos_agg,
    on="SK_ID_CURR",
    how="left"
)

In [63]:
print(master["SK_ID_CURR"].duplicated().sum())

0


#File: Previous application

In [64]:
previous_application.head()

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [65]:
previous_application.isnull().sum()

SK_ID_PREV                           0
SK_ID_CURR                           0
NAME_CONTRACT_TYPE                   0
AMT_ANNUITY                     372235
AMT_APPLICATION                      0
AMT_CREDIT                           1
AMT_DOWN_PAYMENT                895844
AMT_GOODS_PRICE                 385515
WEEKDAY_APPR_PROCESS_START           0
HOUR_APPR_PROCESS_START              0
FLAG_LAST_APPL_PER_CONTRACT          0
NFLAG_LAST_APPL_IN_DAY               0
RATE_DOWN_PAYMENT               895844
RATE_INTEREST_PRIMARY          1664263
RATE_INTEREST_PRIVILEGED       1664263
NAME_CASH_LOAN_PURPOSE               0
NAME_CONTRACT_STATUS                 0
DAYS_DECISION                        0
NAME_PAYMENT_TYPE                    0
CODE_REJECT_REASON                   0
NAME_TYPE_SUITE                 820405
NAME_CLIENT_TYPE                     0
NAME_GOODS_CATEGORY                  0
NAME_PORTFOLIO                       0
NAME_PRODUCT_TYPE                    0
CHANNEL_TYPE             

In [66]:
# aggregate previous applications
previous_agg = (
    previous_application
    .groupby("SK_ID_CURR")
    .agg(
        previous_app_count=("SK_ID_PREV", "count"),
        previous_credit_mean=("AMT_CREDIT", "mean"),
        previous_credit_sum=("AMT_CREDIT", "sum"),
        previous_application_mean=("AMT_APPLICATION", "mean")
    )
    .reset_index()
)

In [67]:
print(previous_agg.head())

   SK_ID_CURR  previous_app_count  previous_credit_mean  previous_credit_sum  \
0      100001                   1              23787.00              23787.0   
1      100002                   1             179055.00             179055.0   
2      100003                   3             484191.00            1452573.0   
3      100004                   1              20106.00              20106.0   
4      100005                   2              20076.75              40153.5   

   previous_application_mean  
0                   24835.50  
1                  179055.00  
2                  435436.50  
3                   24282.00  
4                   22308.75  


In [68]:
#Merge Previous application
master = master.merge(
    previous_agg,
    on="SK_ID_CURR",
    how="left"
)

In [69]:
print(master["SK_ID_CURR"].duplicated().sum())

0


#Final Master Data checking

In [70]:
print("Rows:", master.shape[0])
print("Columns:", master.shape[1])
print("Unique applicants:", master["SK_ID_CURR"].nunique())
print("Duplicate applicant IDs:", master["SK_ID_CURR"].duplicated().sum())

Rows: 307511
Columns: 152
Unique applicants: 307511
Duplicate applicant IDs: 0


In [71]:
missing = (master.isnull().mean().sort_values(ascending=False) * 100)

print(missing.head(20))

cc_limit_mean               71.739222
cc_balance_mean             71.739222
cc_balance_max              71.739222
cc_payment_mean             71.739222
cc_record_count             71.739222
COMMONAREA_MODE             69.872297
COMMONAREA_AVG              69.872297
COMMONAREA_MEDI             69.872297
NONLIVINGAPARTMENTS_MEDI    69.432963
NONLIVINGAPARTMENTS_MODE    69.432963
NONLIVINGAPARTMENTS_AVG     69.432963
FONDKAPREMONT_MODE          68.386172
LIVINGAPARTMENTS_MEDI       68.354953
LIVINGAPARTMENTS_AVG        68.354953
LIVINGAPARTMENTS_MODE       68.354953
FLOORSMIN_AVG               67.848630
FLOORSMIN_MODE              67.848630
FLOORSMIN_MEDI              67.848630
YEARS_BUILD_MEDI            66.497784
YEARS_BUILD_MODE            66.497784
dtype: float64


In [72]:
#print(master.columns.tolist())
for i, col in enumerate(master.columns, start=1):
    print(i, col)

1 SK_ID_CURR
2 TARGET
3 NAME_CONTRACT_TYPE
4 CODE_GENDER
5 FLAG_OWN_CAR
6 FLAG_OWN_REALTY
7 CNT_CHILDREN
8 AMT_INCOME_TOTAL
9 AMT_CREDIT
10 AMT_ANNUITY
11 AMT_GOODS_PRICE
12 NAME_TYPE_SUITE
13 NAME_INCOME_TYPE
14 NAME_EDUCATION_TYPE
15 NAME_FAMILY_STATUS
16 NAME_HOUSING_TYPE
17 REGION_POPULATION_RELATIVE
18 DAYS_BIRTH
19 DAYS_EMPLOYED
20 DAYS_REGISTRATION
21 DAYS_ID_PUBLISH
22 OWN_CAR_AGE
23 FLAG_MOBIL
24 FLAG_EMP_PHONE
25 FLAG_WORK_PHONE
26 FLAG_CONT_MOBILE
27 FLAG_PHONE
28 FLAG_EMAIL
29 OCCUPATION_TYPE
30 CNT_FAM_MEMBERS
31 REGION_RATING_CLIENT
32 REGION_RATING_CLIENT_W_CITY
33 WEEKDAY_APPR_PROCESS_START
34 HOUR_APPR_PROCESS_START
35 REG_REGION_NOT_LIVE_REGION
36 REG_REGION_NOT_WORK_REGION
37 LIVE_REGION_NOT_WORK_REGION
38 REG_CITY_NOT_LIVE_CITY
39 REG_CITY_NOT_WORK_CITY
40 LIVE_CITY_NOT_WORK_CITY
41 ORGANIZATION_TYPE
42 EXT_SOURCE_1
43 EXT_SOURCE_2
44 EXT_SOURCE_3
45 APARTMENTS_AVG
46 BASEMENTAREA_AVG
47 YEARS_BEGINEXPLUATATION_AVG
48 YEARS_BUILD_AVG
49 COMMONAREA_AVG
50 ELEVATORS_A

In [73]:
for i, col in enumerate(application_test.columns, start=1):
    print(i, col)

1 SK_ID_CURR
2 NAME_CONTRACT_TYPE
3 CODE_GENDER
4 FLAG_OWN_CAR
5 FLAG_OWN_REALTY
6 CNT_CHILDREN
7 AMT_INCOME_TOTAL
8 AMT_CREDIT
9 AMT_ANNUITY
10 AMT_GOODS_PRICE
11 NAME_TYPE_SUITE
12 NAME_INCOME_TYPE
13 NAME_EDUCATION_TYPE
14 NAME_FAMILY_STATUS
15 NAME_HOUSING_TYPE
16 REGION_POPULATION_RELATIVE
17 DAYS_BIRTH
18 DAYS_EMPLOYED
19 DAYS_REGISTRATION
20 DAYS_ID_PUBLISH
21 OWN_CAR_AGE
22 FLAG_MOBIL
23 FLAG_EMP_PHONE
24 FLAG_WORK_PHONE
25 FLAG_CONT_MOBILE
26 FLAG_PHONE
27 FLAG_EMAIL
28 OCCUPATION_TYPE
29 CNT_FAM_MEMBERS
30 REGION_RATING_CLIENT
31 REGION_RATING_CLIENT_W_CITY
32 WEEKDAY_APPR_PROCESS_START
33 HOUR_APPR_PROCESS_START
34 REG_REGION_NOT_LIVE_REGION
35 REG_REGION_NOT_WORK_REGION
36 LIVE_REGION_NOT_WORK_REGION
37 REG_CITY_NOT_LIVE_CITY
38 REG_CITY_NOT_WORK_CITY
39 LIVE_CITY_NOT_WORK_CITY
40 ORGANIZATION_TYPE
41 EXT_SOURCE_1
42 EXT_SOURCE_2
43 EXT_SOURCE_3
44 APARTMENTS_AVG
45 BASEMENTAREA_AVG
46 YEARS_BEGINEXPLUATATION_AVG
47 YEARS_BUILD_AVG
48 COMMONAREA_AVG
49 ELEVATORS_AVG
50 ENTR

In [74]:
File: sample_submission

In [75]:
sample_submission.head()

,SK_ID_CURR,TARGET
0,100001,0.5
1,100005,0.5
2,100013,0.5
3,100028,0.5
4,100038,0.5


In [76]:
sample_submission.shape

(48744, 2)

In [77]:
application_test.shape

(48744, 121)

In [78]:
sample_submission.duplicated().sum()

np.int64(0)

In [79]:
print(master.shape)

(307511, 152)


# Checking master data

In [101]:
print(master.info)
print(master.duplicated().sum())
print(master.head())

<bound method DataFrame.info of         SK_ID_CURR  TARGET NAME_CONTRACT_TYPE CODE_GENDER FLAG_OWN_CAR  \
0           100002       1         Cash loans           M            N   
1           100003       0         Cash loans           F            N   
2           100004       0    Revolving loans           M            Y   
3           100006       0         Cash loans           F            N   
4           100007       0         Cash loans           M            N   
...            ...     ...                ...         ...          ...   
307506      456251       0         Cash loans           M            N   
307507      456252       0         Cash loans           F            N   
307508      456253       0         Cash loans           F            N   
307509      456254       1         Cash loans           F            N   
307510      456255       0         Cash loans           F            N   

       FLAG_OWN_REALTY  CNT_CHILDREN  AMT_INCOME_TOTAL  AMT_CREDIT  \
0        

In [102]:
print(master.isnull())

        SK_ID_CURR  TARGET  NAME_CONTRACT_TYPE  CODE_GENDER  FLAG_OWN_CAR  \
0            False   False               False        False         False   
1            False   False               False        False         False   
2            False   False               False        False         False   
3            False   False               False        False         False   
4            False   False               False        False         False   
...            ...     ...                 ...          ...           ...   
307506       False   False               False        False         False   
307507       False   False               False        False         False   
307508       False   False               False        False         False   
307509       False   False               False        False         False   
307510       False   False               False        False         False   

        FLAG_OWN_REALTY  CNT_CHILDREN  AMT_INCOME_TOTAL  AMT_CREDIT  \
0   

In [103]:
print(master["SK_ID_CURR"].duplicated().sum())

0


In [105]:
# Total missing values
master.isnull().sum().sum()

np.int64(10841124)

In [106]:
#Missing values
master.isnull().sum()

SK_ID_CURR                       0
TARGET                           0
NAME_CONTRACT_TYPE               0
CODE_GENDER                      0
FLAG_OWN_CAR                     0
                             ...  
pos_dpd_def_max              18067
previous_app_count           16454
previous_credit_mean         16454
previous_credit_sum          16454
previous_application_mean    16454
Length: 152, dtype: int64

In [108]:
#Ony Column with missing
missing = master.isnull().sum()

print(missing[missing > 0].sort_values(ascending=False))

cc_payment_mean           220606
cc_record_count           220606
cc_balance_max            220606
cc_balance_mean           220606
cc_limit_mean             220606
                           ...  
EXT_SOURCE_2                 660
AMT_GOODS_PRICE              278
AMT_ANNUITY                   12
CNT_FAM_MEMBERS                2
DAYS_LAST_PHONE_CHANGE         1
Length: 97, dtype: int64


In [109]:
#quick check on master
print("Master shape:", master.shape)
print("Total missing values:", master.isnull().sum().sum())
print("Duplicate rows:", master.duplicated().sum())
print("Duplicate SK_ID_CURR:", master["SK_ID_CURR"].duplicated().sum())

Master shape: (307511, 152)
Total missing values: 10841124
Duplicate rows: 0
Duplicate SK_ID_CURR: 0


In [110]:
#Investigate missing values
missing_summary = pd.DataFrame({
    "Missing_Count": master.isnull().sum(),
    "Missing_Percentage": master.isnull().mean() * 100
})

missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values("Missing_Percentage", ascending=False)

print(missing_summary)

                        Missing_Count  Missing_Percentage
cc_payment_mean                220606           71.739222
cc_record_count                220606           71.739222
cc_balance_max                 220606           71.739222
cc_balance_mean                220606           71.739222
cc_limit_mean                  220606           71.739222
...                               ...                 ...
EXT_SOURCE_2                      660            0.214626
AMT_GOODS_PRICE                   278            0.090403
AMT_ANNUITY                        12            0.003902
CNT_FAM_MEMBERS                     2            0.000650
DAYS_LAST_PHONE_CHANGE              1            0.000325

[97 rows x 2 columns]


In [112]:
print((master.isnull().sum() > 0).sum())

97


In [113]:
print((master.isnull().sum() == 0).sum())

55


In [126]:
print(application_train.shape)
print(master.shape)
print(application_test.shape)

(307511, 122)
(307511, 152)
(48744, 121)


In [116]:
#Evaluating additional columns
new_columns = [
    col for col in master.columns
    if col not in application_train.columns
]

print(len(new_columns))
print(new_columns)

30
['bureau_month_count', 'bureau_loan_count', 'bureau_credit_sum', 'bureau_credit_mean', 'bureau_debt_sum', 'bureau_overdue_sum', 'cc_record_count', 'cc_balance_mean', 'cc_balance_max', 'cc_limit_mean', 'cc_payment_mean', 'installment_count', 'payment_total', 'payment_mean', 'payment_max', 'installment_amount_mean', 'payment_delay_mean', 'payment_delay_max', 'pos_record_count', 'pos_months_count', 'pos_installment_mean', 'pos_future_installment_mean', 'pos_dpd_mean', 'pos_dpd_max', 'pos_dpd_def_mean', 'pos_dpd_def_max', 'previous_app_count', 'previous_credit_mean', 'previous_credit_sum', 'previous_application_mean']


In [117]:
master[new_columns].isnull().sum().sort_values(ascending=False)

cc_balance_mean                220606
cc_payment_mean                220606
cc_limit_mean                  220606
cc_balance_max                 220606
cc_record_count                220606
bureau_credit_mean              44021
bureau_overdue_sum              44020
bureau_debt_sum                 44020
bureau_credit_sum               44020
bureau_loan_count               44020
bureau_month_count              44020
pos_installment_mean            18091
pos_future_installment_mean     18091
pos_record_count                18067
pos_dpd_def_max                 18067
pos_dpd_def_mean                18067
pos_dpd_max                     18067
pos_months_count                18067
pos_dpd_mean                    18067
previous_app_count              16454
previous_credit_sum             16454
previous_credit_mean            16454
previous_application_mean       16454
payment_delay_max               15876
payment_mean                    15876
payment_max                     15876
payment_dela

In [118]:
(master[new_columns].isnull().mean() * 100).sort_values(ascending=False)

cc_balance_mean                71.739222
cc_payment_mean                71.739222
cc_limit_mean                  71.739222
cc_balance_max                 71.739222
cc_record_count                71.739222
bureau_credit_mean             14.315260
bureau_overdue_sum             14.314935
bureau_debt_sum                14.314935
bureau_credit_sum              14.314935
bureau_loan_count              14.314935
bureau_month_count             14.314935
pos_installment_mean            5.883042
pos_future_installment_mean     5.883042
pos_record_count                5.875237
pos_dpd_def_max                 5.875237
pos_dpd_def_mean                5.875237
pos_dpd_max                     5.875237
pos_months_count                5.875237
pos_dpd_mean                    5.875237
previous_app_count              5.350703
previous_credit_sum             5.350703
previous_credit_mean            5.350703
previous_application_mean       5.350703
payment_delay_max               5.162742
payment_mean    

In [119]:
master["cc_record_count"].describe()

count    86905.000000
mean        37.143605
std         33.698512
min          1.000000
25%         10.000000
50%         22.000000
75%         75.000000
max        192.000000
Name: cc_record_count, dtype: float64

In [120]:
master["cc_record_count"].value_counts(dropna=False).head(10)

cc_record_count
NaN     220606
96.0      6153
8.0       4032
7.0       4004
9.0       3529
10.0      3269
6.0       3250
11.0      2863
95.0      2717
12.0      2332
Name: count, dtype: int64

In [125]:
# To make csv file 
os.makedirs("data/processed", exist_ok=True)

master.to_csv("data/processed/master.csv", index=False)